In [1]:
from src.egnn_tda.dataset import *
from src.egnn_tda.models import *
from src.egnn_tda.train import *
from src.egnn_tda.pi_plot import *

import torch
import src.templates
import plotly.io as pio
import plotly.graph_objects as go
pio.templates.default = "template_TNR"
from plotly.subplots import make_subplots

C:\Users\alwas\Desktop\egnn_tda\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = QM9Dataset(root="./dataset/qm9_v1.3_total",
                    transform_list = None,
                    pre_transform_list = [TDA_transform(), RDKitAromaticSPPreTransform(), Add_node_attrs(), DropFields("smiles", "name", "z", "x")],
                    pre_filter_list = [MaxAtomsFilter(max_atoms= 1000)],
                    force_reload = False
)

In [6]:
visualize_pi(dataset[75835].pi[0])


In [48]:
ckpt = torch.load("../checkpoints/model_gnn_v1.3_total.pt", map_location="cpu")
model = EGNN(node_attr_dim = dataset.node_attr.shape[1],
              edge_attr_dim   = dataset.edge_attr.shape[1],
              hidden_dim = 64,
              num_layers = 7,
              equivariant=False
              )
model.load_state_dict(ckpt["model_state"])
y_mean = ckpt["mean_y"]

In [49]:
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import torch

n = len(dataset)
n_train = int(0.90 * n)
n_test = n - n_train
train_ds, test_ds = random_split(dataset, [n_train, n_test], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

In [50]:
model.to("cpu")
MAE_train = []
MAE_valid = []

dataset_train = []
dataset_valid = []
egnn_train = []
egnn_valid = []

for batch in tqdm(train_loader):
     egnn_train.append(model(batch).detach() + y_mean.detach().cpu().numpy())
     MAE_train.append( (np.abs(model(batch).detach() + y_mean.detach().cpu().numpy() -  batch.y)).numpy() )
     dataset_train.append(batch.y.detach().cpu().numpy())

for batch in tqdm(test_loader):
    egnn_valid.append(model(batch).detach() + y_mean.detach().cpu().numpy())
    MAE_valid.append( (np.abs(model(batch).detach() + y_mean.detach().cpu().numpy() -  batch.y)).numpy() )
    dataset_valid.append(batch.y.detach().cpu().numpy())

  0%|          | 0/1840 [00:00<?, ?it/s]C:\Users\alwas\AppData\Local\Temp\ipykernel_13368\2906209859.py:11: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)

C:\Users\alwas\AppData\Local\Temp\ipykernel_13368\2906209859.py:12: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)

  0%|          | 0/205 [00:00<?, ?it/s]C:\Users\alwas\AppData\Local\Temp\ipykernel_13368\2906209859.py:16: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)

C:\Users\alwas\AppData\Local\Temp\ipykernel_13368\2906209859.py:17: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)

100%|██████████| 205/205 [00:09<00:00, 22.24it/s]


In [51]:
import src.templates
import plotly.io as pio
import plotly.graph_objects as go
pio.templates.default = "template_TNR"



fig = go.Figure(layout={
        'plot_bgcolor': 'white',
        'paper_bgcolor' : 'white',})

fig.update_layout(width = 600,
                  height = 600
                  )
fig.update_xaxes(title = "EGNN_TDA (eV)",
                 range=[0, 15]
)
fig.update_yaxes(title = "Dataset (eV)",
                 nticks=5,
                 range=[0, 15])

fig.add_trace(go.Scatter(x = [-15, 15], y = [-15, 15], mode = 'lines', line=dict(color = "lightgrey"), showlegend=False))

fig.add_trace(go.Scatter(x = np.concatenate(egnn_train),
                         y = np.concatenate(dataset_train),
                         mode = 'markers',
                         marker = dict(color = "blue"),
                         showlegend=False,
                         )
              )


fig.add_trace(go.Scatter(x = np.concatenate(egnn_valid),
                         y = np.concatenate(dataset_valid),
                         mode = 'markers',
                         marker = dict(color = "green"),
                         showlegend=False,
                         )
                  )

fig.show()
fig.write_image("egnn_tda_acc.png", scale =2)
print(f"MAE (train): {1000 * np.mean(np.concatenate(MAE_train)):.2f} meV")
print(f"MAE (valid): {1000 * np.mean(np.concatenate(MAE_valid)):.2f} meV")

MAE (train): 91.49 meV
MAE (valid): 90.03 meV
